In [ ]:
# CÉLULA 1: INSTALAÇÃO E SETUP
!pip install osmnx geopandas geopy pandas pyarrow folium shapely

import os
import time
import pandas as pd
import osmnx as ox
from geopy.geocoders import Nominatim, ArcGIS
import folium
import warnings

warnings.filterwarnings('ignore')

# Deleta o arquivo antigo se existir para forçar a nova arquitetura
if os.path.exists('dataset_espacial_rc.parquet'):
    os.remove('dataset_espacial_rc.parquet')
    print("Cache antigo deletado.")

print("Motores de geoprocessamento carregados com sucesso!")

Cache antigo deletado.
Motores de geoprocessamento carregados com sucesso!


In [ ]:
# CÉLULA 2: DIMENSÃO ESTÁTICA DE SAÚDE
# Lista absoluta extraída do portal, já limpa para os motores de satélite
unidades_saude_rc = [
    {"nome": "Vigilância Sanitária", "categoria": "Vigilância Sanitária", "endereco": "Avenida 1, 759, Rio Claro, SP"},
    {"nome": "PSMI - Nossa Senhora de Lourdes", "categoria": "Pronto-Atendimento", "endereco": "Avenida 15, Centro, Rio Claro, SP"},
    {"nome": "Pronto Atendimento Ginecológico", "categoria": "Pronto-Atendimento", "endereco": "Avenida 15, Centro, Rio Claro, SP"},
    {"nome": "UPA 29", "categoria": "Pronto-Atendimento", "endereco": "Avenida 29, 1313, Bairro do Estádio, Rio Claro, SP"},
    {"nome": "UPA Chervezon", "categoria": "Pronto-Atendimento", "endereco": "Rua M-9, 50, Jardim Chervezon, Rio Claro, SP"},
    {"nome": "CAPS III 18 de Maio", "categoria": "Saúde Mental", "endereco": "Rua 15, 442, Consolação, Rio Claro, SP"},
    {"nome": "UBS Jardim Chervezon", "categoria": "UBS", "endereco": "Avenida M-17, 739, Jardim Chervezon, Rio Claro, SP"},
    {"nome": "UBS 29 Oreste Armando", "categoria": "UBS", "endereco": "Avenida 29, 1311, Bairro do Estádio, Rio Claro, SP"},
    {"nome": "UBS Wenzel", "categoria": "UBS", "endereco": "Rua 21, 4219, Wenzel, Rio Claro, SP"},
    {"nome": "UBS Vila Cristina", "categoria": "UBS", "endereco": "Avenida José Felício Castellano, 1784, Rio Claro, SP"},
    {"nome": "USF Assistência", "categoria": "UBS", "endereco": "Avenida 1, Assistência, Rio Claro, SP"},
    {"nome": "USF Ferraz", "categoria": "UBS", "endereco": "Avenida 4, 385, Distrito de Ferraz, Rio Claro, SP"},
    {"nome": "USF Nosso Teto / Boa Vista", "categoria": "UBS", "endereco": "Avenida 88, 147, Jardim Boa Vista, Rio Claro, SP"},
    {"nome": "USF Ajapi", "categoria": "UBS", "endereco": "Rua 4, Ajapi, Rio Claro, SP"},
    {"nome": "USF Mãe Preta I/II", "categoria": "UBS", "endereco": "Rua 12, 300, Mãe Preta, Rio Claro, SP"},
    {"nome": "USF Palmeiras", "categoria": "UBS", "endereco": "Rua 8, 1102, Jardim das Palmeiras, Rio Claro, SP"},
    {"nome": "USF Jardim Novo I e II", "categoria": "UBS", "endereco": "Rua 8, 1012, Jardim Novo I, Rio Claro, SP"},
    {"nome": "USF Benjamin de Castro", "categoria": "UBS", "endereco": "Avenida 8, 420, Jardim Centenário, Rio Claro, SP"},
    {"nome": "USF Bonsucesso / Novo Wenzel", "categoria": "UBS", "endereco": "Rua 6, 680, Jardim Novo Wenzel, Rio Claro, SP"},
    {"nome": "USF Jardim das Flores", "categoria": "UBS", "endereco": "Avenida M-51, Jardim das Flores, Rio Claro, SP"},
    {"nome": "USF Guanabara", "categoria": "UBS", "endereco": "Rua 9, Jardim Guanabara, Rio Claro, SP"},
    {"nome": "USF Panorama", "categoria": "UBS", "endereco": "Avenida 64-PA, 1390, Jardim Panorama, Rio Claro, SP"},
    {"nome": "USF Terra Nova", "categoria": "UBS", "endereco": "Avenida Marginal, 1043, Jardim Terra Nova, Rio Claro, SP"}
]
print(f"{len(unidades_saude_rc)} Unidades de Saúde carregadas na memória.")

23 Unidades de Saúde carregadas na memória.


In [ ]:
# CÉLULA 3: EXTRAÇÃO MASSIVA E GEOCODIFICAÇÃO DUPLA
dados_totais = []
CIDADE = "Rio Claro, São Paulo, Brazil"

# Inicializando Motores
nom_geolocator = Nominatim(user_agent="modelo_preditivo_rc")
arc_geolocator = ArcGIS() # Fallback altamente preciso para o Brasil

print("[1/2] Mapeando Unidades de Saúde (Dual-Engine)...")
for unidade in unidades_saude_rc:
    endereco = unidade['endereco']
    nome = unidade['nome']
    print(f"Buscando: {nome}...", end=" ")

    try:
        # TENTATIVA 1: Nominatim
        location = nom_geolocator.geocode(endereco)
        time.sleep(1)

        if location:
            lat, lon = location.latitude, location.longitude
            print("[OK - OSM]")
        else:
            # TENTATIVA 2: ArcGIS (Perfeito para ruas complexas)
            location = arc_geolocator.geocode(endereco)
            if location:
                lat, lon = location.latitude, location.longitude
                print("[OK - ArcGIS]")
            else:
                print("[FALHA - Inserindo Centro da Cidade como Fallback]")
                lat, lon = -22.4149, -47.5616 # Centro de Rio Claro

        dados_totais.append({
            'nome': nome,
            'categoria': unidade['categoria'],
            'endereco': endereco,
            'latitude': lat,
            'longitude': lon
        })
    except Exception as e:
        print(f"[ERRO DE CONEXÃO]")

print("\n[2/2] Extraindo Infraestrutura Urbana (OpenStreetMap)...")
mapa_tags = {
    "Educação": {'amenity': ['school', 'kindergarten', 'college']},
    "Esporte e Lazer": {'leisure': ['fitness_station', 'sports_centre', 'pitch', 'park']},
    "Alimentação - Restaurante/Fast-food": {'amenity': ['restaurant', 'fast_food', 'cafe', 'food_court']},
    "Alimentação - Mercado": {'shop': ['supermarket', 'convenience', 'wholesale']}
}

for categoria, tags in mapa_tags.items():
    print(f" -> Baixando: {categoria}...")
    try:
        gdf = ox.features_from_place(CIDADE, tags=tags)
        gdf = gdf.to_crs(epsg=4326)
        gdf['lat'] = gdf.geometry.centroid.y
        gdf['lon'] = gdf.geometry.centroid.x

        for _, row in gdf.iterrows():
            dados_totais.append({
                'nome': row.get('name', 'Desconhecido') if pd.notna(row.get('name')) else 'Desconhecido',
                'categoria': categoria,
                'endereco': "Localizado via OSM",
                'latitude': row['lat'],
                'longitude': row['lon']
            })
    except Exception as e:
        print(f"    Aviso: Falha ao baixar {categoria}.")

# Consolidação Final
print("\nGerando base Parquet otimizada...")
df = pd.DataFrame(dados_totais)
df = df.dropna(subset=['latitude', 'longitude'])
df = df.drop_duplicates(subset=['nome', 'latitude', 'longitude'])
df.to_parquet('dataset_espacial_rc.parquet', engine='pyarrow', compression='snappy')
print(f"VITÓRIA! {len(df)} pontos validados e salvos com sucesso.")

[1/2] Mapeando Unidades de Saúde (Dual-Engine)...
Buscando: Vigilância Sanitária... [OK - OSM]
Buscando: PSMI - Nossa Senhora de Lourdes... [OK - OSM]
Buscando: Pronto Atendimento Ginecológico... [OK - OSM]
Buscando: UPA 29... [OK - OSM]
Buscando: UPA Chervezon... [OK - OSM]
Buscando: CAPS III 18 de Maio... [OK - OSM]
Buscando: UBS Jardim Chervezon... [OK - OSM]
Buscando: UBS 29 Oreste Armando... [OK - OSM]
Buscando: UBS Wenzel... [OK - OSM]
Buscando: UBS Vila Cristina... [OK - OSM]
Buscando: USF Assistência... [OK - OSM]
Buscando: USF Ferraz... [OK - ArcGIS]
Buscando: USF Nosso Teto / Boa Vista... [OK - OSM]
Buscando: USF Ajapi... [OK - OSM]
Buscando: USF Mãe Preta I/II... [OK - OSM]
Buscando: USF Palmeiras... [OK - OSM]
Buscando: USF Jardim Novo I e II... [OK - OSM]
Buscando: USF Benjamin de Castro... [OK - OSM]
Buscando: USF Bonsucesso / Novo Wenzel... [OK - OSM]
Buscando: USF Jardim das Flores... [OK - OSM]
Buscando: USF Guanabara... [OK - OSM]
Buscando: USF Panorama... [OK - OSM]


In [ ]:
# CÉLULA 4: RENDERIZAÇÃO DO MAPA VISUAL
df_mapa = pd.read_parquet('dataset_espacial_rc.parquet')

centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()
mapa_rc = folium.Map(location=[centro_lat, centro_lon], zoom_start=13, tiles='OpenStreetMap')

cores = {
    "UBS": "#e74c3c",                             # Vermelho
    "Pronto-Atendimento": "#8e44ad",              # Roxo Escuro
    "Saúde Mental": "#f39c12",                    # Laranja
    "Vigilância Sanitária": "#c0392b",            # Vermelho Sangue
    "Educação": "#3498db",                        # Azul
    "Esporte e Lazer": "#2ecc71",                 # Verde
    "Alimentação - Restaurante/Fast-food": "#e67e22", # Laranja Claro
    "Alimentação - Mercado": "#9b59b6"            # Roxo Claro
}

for index, row in df_mapa.iterrows():
    cat = row['categoria']
    cor_ponto = cores.get(cat, "#34495e")

    # Destaca infraestrutura governamental
    is_gov = cat in ["UBS", "Pronto-Atendimento", "Saúde Mental", "Vigilância Sanitária"]
    raio_ponto = 8 if is_gov else 4
    opacidade = 1.0 if is_gov else 0.5

    html_popup = f"""
    <div style='min-width: 150px'>
        <b>{row['nome']}</b><br>
        <span style='color: gray; font-size: 12px'>{cat}</span>
    </div>
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=raio_ponto,
        popup=folium.Popup(html_popup, max_width=300),
        tooltip=row['nome'],
        color=cor_ponto,
        weight=1,
        fill=True,
        fill_color=cor_ponto,
        fill_opacity=opacidade
    ).add_to(mapa_rc)

mapa_rc

In [ ]:
import pandas as pd
import folium
import time
import osmnx as ox
import matplotlib.colors as mcolors
import random
from geopy.geocoders import ArcGIS
from shapely.geometry import Point, MultiPoint
from shapely.ops import voronoi_diagram

# 1. Lista de dados do seu Excel com "Tradução" para o geocodificador não falhar
dados_excel = [
    {"oficial": "UBS DR ANTONIO RAPHAEL MINERVINO SANTOMAURO", "busca": "USF Boa Vista, Rio Claro, SP"},
    {"oficial": "UBS DR MARIO FITTIPALDI", "busca": "UBS Wenzel, Rio Claro, SP"},
    {"oficial": "UBS DR NICOLINO MAZZIOTTI JARDIM CERVEZAO", "busca": "UBS Jardim Cervezao, Rio Claro, SP"},
    {"oficial": "UBS DR SILVIO ARNALDO PIVA", "busca": "UBS Vila Cristina, Rio Claro, SP"},
    {"oficial": "UBS ORESTES ARMANDO GIOVANNI", "busca": "UBS Bairro do Estadio, Rio Claro, SP"},
    {"oficial": "UBS PSF CELIA AP CECCATO DA SILVA", "busca": "USF Novo Wenzel, Rio Claro, SP"},
    {"oficial": "UBS PSF DA ASSISTENCIA", "busca": "USF Assistencia, Rio Claro, SP"},
    {"oficial": "UBS PSF DE AJAPI FARMACEUTICO ANTONIO GILBERTO FONSECA", "busca": "USF Ajapi, Rio Claro, SP"},
    {"oficial": "UBS PSF DR CELESTINO DONATO", "busca": "USF Guanabara, Rio Claro, SP"},
    {"oficial": "UBS PSF DR DIRCEU FERREIRA PENTEADO", "busca": "USF Jardim Novo, Rio Claro, SP"},
    {"oficial": "UBS PSF DR EMILIO BELTRATI JUNIOR", "busca": "UBS Jardim Bandeirantes, Rio Claro, SP"},
    {"oficial": "UBS PSF DR GILSON GIOVANNI", "busca": "USF Palmeiras, Rio Claro, SP"},
    {"oficial": "UBS PSF DR MOACIR DE OLIVEIRA CAMARGO", "busca": "USF Jardim das Flores, Rio Claro, SP"},
    {"oficial": "UBS PSF DR NORBERTO ANTONIO SIMAO CARNEIRO", "busca": "USF Bela Vista, Rio Claro, SP"}, # Corrigido por proximidade
    {"oficial": "UBS PSF DR OSWALDO AKAMINE", "busca": "USF Panorama, Rio Claro, SP"},
    {"oficial": "UBS PSF JARDIM SANTA ELIZA", "busca": "USF Jardim Santa Eliza, Rio Claro, SP"},
    {"oficial": "UBS PSF PQUE MAE PRETA DR EDUARDO REIS", "busca": "USF Mae Preta, Rio Claro, SP"},
    {"oficial": "USF BELA VISTA DR ARINDAL CARNEIRO CESAR PIRES", "busca": "USF Bela Vista, Rio Claro, SP"},
    {"oficial": "USF JARDIM BRASILIA NEUSA MARIA MORTARI", "busca": "USF Jardim Brasilia, Rio Claro, SP"},
    {"oficial": "USF JARDIM SAO MIGUEL JORCELINO QUINTINO DE FARIA", "busca": "USF Jardim Sao Miguel, Rio Claro, SP"},
    {"oficial": "USF JD PROGRESSO JOSE CARLOS DA SILVA", "busca": "USF Jardim Progresso, Rio Claro, SP"},
]

print("Iniciando geocodificação da base do Excel...")
geolocator = ArcGIS() # Satélite mais forte e permissivo para endereços do Brasil
pontos_validos = []

# 2. Geocodificando cada UBS da lista
for item in dados_excel:
    try:
        loc = geolocator.geocode(item['busca'])
        if loc:
            pontos_validos.append({
                'nome': item['oficial'],
                'lat': loc.latitude,
                'lon': loc.longitude,
                'geometry': Point(loc.longitude, loc.latitude) # Shapely usa (X,Y) -> (Lon,Lat)
            })
    except Exception as e:
        print(f"Erro ao buscar {item['oficial']}")

print(f"Geocodificados {len(pontos_validos)} postos com sucesso.")

# 3. Baixando a Fronteira de Rio Claro para recortar o mapa (OSMnx)
print("Baixando polígono territorial de Rio Claro...")
limite_cidade = ox.geocode_to_gdf("Rio Claro, São Paulo, Brazil").geometry.iloc[0]

# 4. Criando as Regiões Matemáticas (Voronoi)
print("Calculando Áreas de Abrangência (Voronoi)...")
coords_multi = MultiPoint([p['geometry'] for p in pontos_validos])
diagrama_voronoi = voronoi_diagram(coords_multi)

# 5. Desenhando o Mapa Visual
centro_lat = sum([p['lat'] for p in pontos_validos]) / len(pontos_validos)
centro_lon = sum([p['lon'] for p in pontos_validos]) / len(pontos_validos)

mapa_regioes = folium.Map(location=[centro_lat, centro_lon], zoom_start=12, tiles='CartoDB positron')

# Uma paleta de cores para os bairros
cores_disponiveis = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.CSS4_COLORS.values())
random.seed(42) # Mantém as cores sempre iguais nas execuções

# 6. Mapeando e Recortando os Polígonos
# Tentamos associar cada polígono gerado com o ponto (UBS) que está dentro dele
for poly in diagrama_voronoi.geoms:
    # Recorta o polígono infinito pelo limite da cidade
    poly_cortado = poly.intersection(limite_cidade)

    if poly_cortado.is_empty:
        continue

    # Descobre qual UBS pertence a esse polígono específico
    ubs_dona_do_pedaco = "Área Indefinida"
    for p in pontos_validos:
        if poly.contains(p['geometry']):
            ubs_dona_do_pedaco = p['nome']
            break

    # Adiciona a "fatia do bairro" ao mapa
    cor = random.choice(cores_disponiveis)
    folium.GeoJson(
        poly_cortado,
        style_function=lambda x, fillColor=cor: {
            'fillColor': fillColor,
            'color': 'black',
            'weight': 1.5,
            'fillOpacity': 0.35
        },
        tooltip=f"Área de Atendimento:<br><b>{ubs_dona_do_pedaco}</b>"
    ).add_to(mapa_regioes)

# 7. Adicionando as Bolinhas (Marcadores) das UBSs por cima das regiões
for p in pontos_validos:
    folium.CircleMarker(
        location=[p['lat'], p['lon']],
        radius=6,
        color='red',
        fill=True,
        fill_color='darkred',
        fill_opacity=1,
        tooltip=f"📌 {p['nome']}"
    ).add_to(mapa_regioes)

print("Mapa de Áreas de Influência gerado!")
mapa_regioes

Iniciando geocodificação da base do Excel...


Geocodificados 21 postos com sucesso.
Baixando polígono territorial de Rio Claro...
Calculando Áreas de Abrangência (Voronoi)...
Mapa de Áreas de Influência gerado!


In [ ]:
# CÉLULA 4: MAPA DE REGIÕES COM TODOS OS PINGS E TAG DE ABRANGÊNCIA
import pandas as pd
import folium
from shapely.geometry import Point

# 1. Carregar os dados completos (POIs) e os limites da cidade
df_todos_pontos = pd.read_parquet('dataset_espacial_rc.parquet')

# 2. Criar o mapa base
centro_lat = sum([p['lat'] for p in pontos_validos]) / len(pontos_validos)
centro_lon = sum([p['lon'] for p in pontos_validos]) / len(pontos_validos)
mapa_regioes = folium.Map(location=[centro_lat, centro_lon], zoom_start=13, tiles='CartoDB positron')

# 3. Preparar as cores das categorias (mesmo padrão do mapa 1)
cores_categorias = {
    "UBS": "#e74c3c",
    "Pronto-Atendimento": "#8e44ad",
    "Saúde Mental": "#f39c12",
    "Vigilância Sanitária": "#c0392b",
    "Educação": "#3498db",
    "Esporte e Lazer": "#2ecc71",
    "Alimentação - Restaurante/Fast-food": "#e67e22",
    "Alimentação - Mercado": "#9b59b6"
}

# 4. Processar os Polígonos de Voronoi e armazená-los com seus respectivos nomes
regioes_mapeadas = []
for poly in diagrama_voronoi.geoms:
    poly_cortado = poly.intersection(limite_cidade)
    if poly_cortado.is_empty:
        continue

    # Identifica qual UBS é dona desta região
    nome_ubs_regiao = "Área Externa/Indefinida"
    for p in pontos_validos:
        if poly.contains(p['geometry']):
            nome_ubs_regiao = p['nome']
            break

    regioes_mapeadas.append({'poligono': poly_cortado, 'ubs': nome_ubs_regiao})

    # Desenha o polígono no mapa (Fundo colorido)
    folium.GeoJson(
        poly_cortado,
        style_function=lambda x: {
            'fillColor': 'gray', # Cor neutra para o fundo para não conflitar com os pings
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.1
        }
    ).add_to(mapa_rc) # Adicionando ao mapa_regioes (corrigido abaixo)

# Reiniciando o desenho para garantir ordem de camadas
mapa_regioes = folium.Map(location=[centro_lat, centro_lon], zoom_start=13, tiles='CartoDB positron')

# Desenha as bordas das regiões primeiro
for reg in regioes_mapeadas:
    folium.GeoJson(
        reg['poligono'],
        style_function=lambda x: {'fillColor': '#f8f9fa', 'color': '#34495e', 'weight': 2, 'fillOpacity': 0.2},
        tooltip=f"Região de Abrangência: {reg['ubs']}"
    ).add_to(mapa_regioes)

# 5. Iterar sobre TODOS os pontos e descobrir a região de cada um
print(f"Cruzando {len(df_todos_pontos)} pontos com as regiões de saúde...")

for index, row in df_todos_pontos.iterrows():
    ponto_geom = Point(row['longitude'], row['latitude'])
    regiao_do_ponto = "Não identificada"

    # Verifica em qual polígono o ping caiu
    for reg in regioes_mapeadas:
        if reg['poligono'].contains(ponto_geom):
            regiao_do_ponto = reg['ubs']
            break

    cat = row['categoria']
    cor_ponto = cores_categorias.get(cat, "#34495e")

    # Diferencia visualmente saúde de comércio/educação
    is_saude = cat in ["UBS", "Pronto-Atendimento", "Saúde Mental", "Vigilância Sanitária"]
    raio = 7 if is_saude else 4

    # Tag HTML com a nova informação da Região
    html_popup = f"""
    <div style='min-width: 200px; font-family: sans-serif;'>
        <b style='color: {cor_ponto}'>{row['nome']}</b><br>
        <b>Tipo:</b> {cat}<br>
        <hr style='margin: 5px 0;'>
        <b style='color: #2c3e50'>📍 Região NutriAlerta:</b><br>
        {regiao_do_ponto}
    </div>
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=raio,
        popup=folium.Popup(html_popup, max_width=300),
        tooltip=f"{row['nome']} ({regiao_do_ponto})",
        color=cor_ponto,
        weight=1,
        fill=True,
        fill_color=cor_ponto,
        fill_opacity=0.8
    ).add_to(mapa_regioes)

print("Mapa completo gerado com sucesso!")
mapa_regioes

Cruzando 454 pontos com as regiões de saúde...
Mapa completo gerado com sucesso!
